# 03 - Evaluacion causal con ground truth real

Este notebook usa la primera anotacion oracional real de `realtime/`: la fuente `CHARLA SOBRE EL AMOR Y EL DESAMOR`.

Aca ya no medimos casos sinteticos ni clips aislados. Medimos si el cierre causal commitea en el clip donde una oracion realmente termina.

## 1. Setup

Se usa un JSON de ground truth chico y versionable. No se abren videos, ROIs `.npz`, checkpoints ni outputs de entrenamiento.

In [1]:
from pathlib import Path
import json
import sys
from collections import Counter
from IPython.display import Markdown, display

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "realtime" / "src").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def trunc(value, n=130):
    text = str(value).replace("\n", " ")
    return text if len(text) <= n else text[: n - 3] + "..."


def _fmt(value):
    if isinstance(value, float):
        return f"{value:.4f}"
    if isinstance(value, (list, tuple)):
        return ", ".join(str(item) for item in value) or "-"
    text = str(value)
    return text.replace("\n", " ").replace("|", "\\|")


def show_table(rows, columns):
    if not rows:
        display(Markdown("_Sin filas para mostrar._"))
        return
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(_fmt(row.get(col, "")) for col in columns) + " |")
    display(Markdown("\n".join([header, sep, *body])))

print(f"Repo detectado: {ROOT.name}")
from realtime.src.provider_factory import make_closure_provider
from realtime.src.secuencias import evaluate_causal_sequence, load_sequence_ground_truth

ground_truth_path = ROOT / "realtime" / "ground_truth" / "charla_amor_desamor.json"
sequence = load_sequence_ground_truth(ground_truth_path)
provider = make_closure_provider("heuristic")

print("Ground truth:", ground_truth_path.relative_to(ROOT))
print("Fuente:", sequence.source_id)
print("Clips:", len(sequence.clips))
print("Oraciones anotadas:", len(sequence.sentences))

Repo detectado: labios-argentos
Ground truth: realtime\ground_truth\charla_amor_desamor.json
Fuente: CHARLA SOBRE EL AMOR Y EL DESAMOR
Clips: 233
Oraciones anotadas: 167


## 2. Que hay anotado

Cada oracion tiene `start_clip`, `end_clip` y `commit_after_clip`. Si una oracion cruza varios clips, el sistema deberia esperar hasta el ultimo clip necesario.

In [2]:
clip_order = {clip.clip_id: clip.order for clip in sequence.clips}
span_rows = []
span_counts = Counter()
for sentence in sequence.sentences:
    span = clip_order[sentence.end_clip] - clip_order[sentence.start_clip] + 1
    span_counts[span] += 1
for span, count in sorted(span_counts.items())[:10]:
    span_rows.append({"clips_por_oracion": span, "cantidad": count})
show_table(span_rows, ["clips_por_oracion", "cantidad"])

sample_sentences = []
for sentence in sequence.sentences[:10]:
    sample_sentences.append({
        "sentence_id": sentence.sentence_id,
        "texto": trunc(sentence.text),
        "start_clip": sentence.start_clip,
        "end_clip": sentence.end_clip,
        "commit_after_clip": sentence.commit_after_clip,
    })
show_table(sample_sentences, ["sentence_id", "texto", "start_clip", "end_clip", "commit_after_clip"])

| clips_por_oracion | cantidad |
| --- | --- |
| 1 | 80 |
| 2 | 66 |
| 3 | 16 |
| 4 | 5 |

| sentence_id | texto | start_clip | end_clip | commit_after_clip |
| --- | --- | --- | --- | --- |
| s001 | voy a antereccionar maria del cerro y ahora que hay un monton de gente en el chat les quiero decir algo. | clip_0000 | clip_0000 | clip_0000 |
| s002 | la vida, aca voy a aparecer un hippie o esas finas que leen el horoscopo, pero la vida posta que se trata de energias. | clip_0001 | clip_0002 | clip_0002 |
| s003 | entienden que estamos haciendo un stream a las 4 de la mañana en españa y tengo 24k de viewers hablando de futbol, algo totalme... | clip_0003 | clip_0004 | clip_0004 |
| s004 | estan saliendo todos los dias los rankings de twitch y a mi, yo ya tengo 10 11 años de streamer, no me interesa ranquear, pero ... | clip_0005 | clip_0009 | clip_0009 |
| s005 | ojo, yo veo el numero y no les voy a negarme, me motivo, me motivo, las cosas estan saliendo bien y se los quiero agradecer, si... | clip_0013 | clip_0015 | clip_0015 |
| s006 | porque yo agradecido por mas que sea 2 3 4 20 50 500 22000, agraecido siempre, pero vale destacar que en un momento critico, po... | clip_0016 | clip_0018 | clip_0018 |
| s007 | ojo, despues de que llueve sale el sol. | clip_0019 | clip_0019 | clip_0019 |
| s008 | ahora solo me falta terminar el año ganando la velada, enamorandome y eventualmente el año que viene planeando tener un hijo. | clip_0020 | clip_0021 | clip_0021 |
| s009 | ya esta, vida cumplida. | clip_0022 | clip_0022 | clip_0022 |
| s010 | me doy vuelta el juego de la vida, no quiero mas nada. | clip_0023 | clip_0023 | clip_0023 |

## 3. Timeline causal esperado

Esta tabla muestra la forma correcta de mirar el problema: en cada clip solo se ve el buffer acumulado hasta ese momento. No hay contexto futuro.

In [3]:
expected_by_clip = {sentence.commit_after_clip: sentence.sentence_id for sentence in sequence.sentences}
buffer = []
timeline = []
for clip in sequence.clips[:28]:
    buffer.append(clip.text)
    expected_action = "commit" if clip.clip_id in expected_by_clip else "wait"
    timeline.append({
        "clip": clip.clip_id,
        "texto_clip": trunc(clip.text, 80),
        "buffer_visible": trunc(" ".join(buffer), 140),
        "esperado": expected_action,
        "sentence_id": expected_by_clip.get(clip.clip_id, ""),
    })
    if expected_action == "commit":
        buffer = []
show_table(timeline, ["clip", "texto_clip", "buffer_visible", "esperado", "sentence_id"])

| clip | texto_clip | buffer_visible | esperado | sentence_id |
| --- | --- | --- | --- | --- |
| clip_0000 | voy a antereccionar maria del cerro y ahora que hay un monton de gente en el ... | voy a antereccionar maria del cerro y ahora que hay un monton de gente en el chat les quiero decir algo | commit | s001 |
| clip_0001 | la vida aca voy a aparecer un hippie o esas finas que leen el horoscopo | la vida aca voy a aparecer un hippie o esas finas que leen el horoscopo | wait |  |
| clip_0002 | pero la vida posta que se trata de energias | la vida aca voy a aparecer un hippie o esas finas que leen el horoscopo pero la vida posta que se trata de energias | commit | s002 |
| clip_0003 | entienden que estamos haciendo un stream a las 4 de la mañana en españa y ten... | entienden que estamos haciendo un stream a las 4 de la mañana en españa y tengo 24k de viewers hablando de futbol | wait |  |
| clip_0004 | algo totalmente impensado estan saliendo todos los dias los rankings de twitc... | entienden que estamos haciendo un stream a las 4 de la mañana en españa y tengo 24k de viewers hablando de futbol algo totalmente impensa... | commit | s003 |
| clip_0005 | yo ya tengo 10 11 años de streamer no me interesa ranquear pero te la blanqueo | yo ya tengo 10 11 años de streamer no me interesa ranquear pero te la blanqueo | wait |  |
| clip_0008 | ah buena poronga 23k ojo me la sube sabe por que me la sube | yo ya tengo 10 11 años de streamer no me interesa ranquear pero te la blanqueo ah buena poronga 23k ojo me la sube sabe por que me la sube | wait |  |
| clip_0009 | porque eso es mi historia por eso estoy donde estoy por eso soy main event | yo ya tengo 10 11 años de streamer no me interesa ranquear pero te la blanqueo ah buena poronga 23k ojo me la sube sabe por que me la sub... | commit | s004 |
| clip_0013 | ojo yo veo el numero y no les voy a negarme me motivo me motivo | ojo yo veo el numero y no les voy a negarme me motivo me motivo | wait |  |
| clip_0014 | las cosas estan saliendo bien y se los quiero agradecer si yo tuviera 5k de v... | ojo yo veo el numero y no les voy a negarme me motivo me motivo las cosas estan saliendo bien y se los quiero agradecer si yo tuviera 5k ... | wait |  |
| clip_0015 | igual diria gracias por mirarme | ojo yo veo el numero y no les voy a negarme me motivo me motivo las cosas estan saliendo bien y se los quiero agradecer si yo tuviera 5k ... | commit | s005 |
| clip_0016 | porque yo agradecido por mas que sea 2 3 4 20 50 500 22000 | porque yo agradecido por mas que sea 2 3 4 20 50 500 22000 | wait |  |
| clip_0017 | agraecido siempre pero vale destacar que en un momento critico porque es un m... | porque yo agradecido por mas que sea 2 3 4 20 50 500 22000 agraecido siempre pero vale destacar que en un momento critico porque es un mo... | wait |  |
| clip_0018 | las cosas estan empezando a salir de nuevo despues de que llueve sale el sol | porque yo agradecido por mas que sea 2 3 4 20 50 500 22000 agraecido siempre pero vale destacar que en un momento critico porque es un mo... | commit | s006 |
| clip_0019 | ojo despues de que llueve sale el sol | ojo despues de que llueve sale el sol | commit | s007 |
| clip_0020 | ahora solo me falta terminar el año ganando la velada enamorandome | ahora solo me falta terminar el año ganando la velada enamorandome | wait |  |
| clip_0021 | y eventualmente el año que viene planeando tener un hijo | ahora solo me falta terminar el año ganando la velada enamorandome y eventualmente el año que viene planeando tener un hijo | commit | s008 |
| clip_0022 | ya esta vida cumplida | ya esta vida cumplida | commit | s009 |
| clip_0023 | me doy vuelta el juego de la vida no quiero mas nada | me doy vuelta el juego de la vida no quiero mas nada | commit | s010 |
| clip_0024 | tres hijos tres o dos un chico y una chica un hijo y una hija minimo | tres hijos tres o dos un chico y una chica un hijo y una hija minimo | commit | s011 |
| clip_0025 | una novia que me ame que me valore que me quiera que me acompañe | una novia que me ame que me valore que me quiera que me acompañe | wait |  |
| clip_0026 | poder acompañarla yo que haga deporte pretensiones bajas no me importa lo demas | una novia que me ame que me valore que me quiera que me acompañe poder acompañarla yo que haga deporte pretensiones bajas no me importa l... | commit | s012 |
| clip_0027 | que si el culo que es esto no me importa no me importa | que si el culo que es esto no me importa no me importa | commit | s013 |
| clip_0028 | se tiene un culito lindo mejor mejor | se tiene un culito lindo mejor mejor | wait |  |
| clip_0029 | pero yo prefiero que haga deporte que tenga culo | se tiene un culito lindo mejor mejor pero yo prefiero que haga deporte que tenga culo | commit | s014 |
| clip_0030 | el culo un año y te cansas te da igual | el culo un año y te cansas te da igual | commit | s015 |
| clip_0031 | pero que haga deporte ir a verla imaginante que juega el futbol | pero que haga deporte ir a verla imaginante que juega el futbol | wait |  |
| clip_0032 | que juega el hockey la vas a ver gritar los goles la bancas | pero que haga deporte ir a verla imaginante que juega el futbol que juega el hockey la vas a ver gritar los goles la bancas | commit | s016 |

## 4. Resultado de la heuristica actual

La heuristica ya no da `1.0`: ahora se la mide contra un ground truth oracional real. Esto sirve para ver donde falla y para justificar un provider entrenado o un LLM.

In [4]:
summary = evaluate_causal_sequence(sequence, provider)
metric_rows = [
    {"metrica": "clips", "valor": summary["clips"]},
    {"metrica": "commits_esperados", "valor": summary["expected_commits"]},
    {"metrica": "commits_predichos", "valor": summary["predicted_commits"]},
    {"metrica": "commits_correctos", "valor": summary["correct_commits"]},
    {"metrica": "commits_tempranos", "valor": summary["early_commits"]},
    {"metrica": "waits_tardios", "valor": summary["late_waits"]},
    {"metrica": "commits_faltantes", "valor": summary["missing_commits"]},
    {"metrica": "low_confidence_inesperado", "valor": summary["unexpected_low_confidence"]},
    {"metrica": "precision_commit", "valor": summary["commit_precision"]},
    {"metrica": "recall_commit", "valor": summary["commit_recall"]},
    {"metrica": "latencia_p50_ms", "valor": summary["latency_ms"]["p50"]},
    {"metrica": "latencia_p95_ms", "valor": summary["latency_ms"]["p95"]},
    {"metrica": "fallbacks", "valor": summary["fallbacks"]},
]
show_table(metric_rows, ["metrica", "valor"])

| metrica | valor |
| --- | --- |
| clips | 233 |
| commits_esperados | 167 |
| commits_predichos | 196 |
| commits_correctos | 129 |
| commits_tempranos | 67 |
| waits_tardios | 28 |
| commits_faltantes | 38 |
| low_confidence_inesperado | 2 |
| precision_commit | 0.6582 |
| recall_commit | 0.7725 |
| latencia_p50_ms | 0.0665 |
| latencia_p95_ms | 0.0903 |
| fallbacks | 0 |

## 5. Errores que explican el problema

Estos ejemplos son utiles para decidir el siguiente paso. Un commit temprano corta una oracion antes de tiempo; un wait tardio deja pasar el punto donde ya se podia cerrar.

In [5]:
early_rows = []
late_rows = []
for row in summary["rows"]:
    if row["predicted_action"] == "commit" and row["expected_action"] != "commit" and len(early_rows) < 8:
        early_rows.append({
            "clip": row["clip_id"],
            "buffer": trunc(row["buffer_text"]),
            "predicho": row["predicted_action"],
            "esperado": row["expected_action"],
            "razon": row["reason"],
        })
    if row["predicted_action"] != "commit" and row["expected_action"] == "commit" and len(late_rows) < 8:
        late_rows.append({
            "clip": row["clip_id"],
            "buffer": trunc(row["buffer_text"]),
            "predicho": row["predicted_action"],
            "esperado": row["expected_action"],
            "sentence_id": row["expected_sentence_id"],
            "razon": row["reason"],
        })

print("Commits tempranos: muestra")
show_table(early_rows, ["clip", "buffer", "predicho", "esperado", "razon"])
print("Waits tardios: muestra")
show_table(late_rows, ["clip", "buffer", "predicho", "esperado", "sentence_id", "razon"])

Commits tempranos: muestra


| clip | buffer | predicho | esperado | razon |
| --- | --- | --- | --- | --- |
| clip_0001 | la vida aca voy a aparecer un hippie o esas finas que leen el horoscopo | commit | wait | contexto_suficiente_sin_conector_colgante |
| clip_0003 | entienden que estamos haciendo un stream a las 4 de la mañana en españa y tengo 24k de viewers hablando de futbol | commit | wait | contexto_suficiente_sin_conector_colgante |
| clip_0005 | yo ya tengo 10 11 años de streamer no me interesa ranquear pero te la blanqueo | commit | wait | contexto_suficiente_sin_conector_colgante |
| clip_0008 | ah buena poronga 23k ojo me la sube sabe por que me la sube | commit | wait | contexto_suficiente_sin_conector_colgante |
| clip_0013 | ojo yo veo el numero y no les voy a negarme me motivo me motivo | commit | wait | contexto_suficiente_sin_conector_colgante |
| clip_0014 | las cosas estan saliendo bien y se los quiero agradecer si yo tuviera 5k de viewers igual estaria motivado | commit | wait | contexto_suficiente_sin_conector_colgante |
| clip_0016 | igual diria gracias por mirarme porque yo agradecido por mas que sea 2 3 4 20 50 500 22000 | commit | wait | contexto_suficiente_sin_conector_colgante |
| clip_0017 | agraecido siempre pero vale destacar que en un momento critico porque es un momento critico para mi | commit | wait | contexto_suficiente_sin_conector_colgante |

Waits tardios: muestra


| clip | buffer | predicho | esperado | sentence_id | razon |
| --- | --- | --- | --- | --- | --- |
| clip_0015 | igual diria gracias por mirarme | wait | commit | s005 | contexto_insuficiente |
| clip_0022 | ya esta vida cumplida | wait | commit | s009 | contexto_insuficiente |
| clip_0034 | todos necesitamos amor | wait | commit | s018 | contexto_insuficiente |
| clip_0042 | cada uno ve pero no se cierren | wait | commit | s023 | contexto_insuficiente |
| clip_0045 | hay que ver donde te sentis mejor | wait | commit | s025 | contexto_insuficiente |
| clip_0049 | van a llorar van a sufrir van a extrañar por que | wait | commit | s028 | conector_colgante |
| clip_0060 | enamorarte lo 18 te quieres enamorar lo 18 no no no | low_confidence | commit | s036 | texto_repetitivo |
| clip_0062 | mandale mandale cumbia | low_confidence | commit | s038 | texto_repetitivo |

## 6. Lectura final

Resultado de este notebook:

- Las heuristicas no son al pedo: dan baseline, fallback seguro y errores interpretables.
- Pero las heuristicas no son suficiente para entrenar por si solas. Lo entrenable de verdad sale del ground truth oracional y del feedback revisado.
- Esta fuente ya permite comparar `heuristic`, `LLM zero-shot` y un futuro `student causal` con las mismas metricas.
- La heuristica actual quedaria como baseline/fallback; el objetivo siguiente es ver si un LLM o un modelo entrenado reducen commits tempranos y waits tardios.